In [1]:
import s3fs
import pandas as pd

fs = s3fs.S3FileSystem(anon=True)

base_path = "oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2025/resstock_amy2018_release_1"

# Load the baseline metadata for California
meta_ca = pd.read_parquet(
    f"s3://{base_path}/metadata_and_annual_results/by_state/full/parquet/state=CA/CA_upgrade0.parquet",
    filesystem=fs
)

print(meta_ca.shape)
print(meta_ca.columns.tolist())
print(meta_ca.head())

(57075, 772)
['bldg_id', 'completed_status', 'upgrade', 'in.upgrade_name', 'weight', 'applicability', 'in.sqft..ft2', 'in.representative_income', 'in.county_name', 'in.ahs_region', 'in.aiannh_area', 'in.area_median_income', 'in.ashrae_iecc_climate_zone_2004', 'in.ashrae_iecc_climate_zone_2004_sub_cz_split', 'in.bathroom_spot_vent_hour', 'in.battery', 'in.bedrooms', 'in.building_america_climate_zone', 'in.cec_climate_zone', 'in.ceiling_fan', 'in.census_division', 'in.census_division_recs', 'in.census_region', 'in.city', 'in.clothes_dryer', 'in.clothes_dryer_usage_level', 'in.clothes_washer', 'in.clothes_washer_presence', 'in.clothes_washer_usage_level', 'in.cooking_range', 'in.cooking_range_usage_level', 'in.cooling_unavailable_days', 'in.cooling_setpoint', 'in.cooling_setpoint_has_offset', 'in.cooling_setpoint_offset_magnitude', 'in.cooling_setpoint_offset_period', 'in.corridor', 'in.county', 'in.county_and_puma', 'in.county_metro_status', 'in.custom_state', 'in.dehumidifier', 'in.dish

In [2]:
import numpy as np
# Option B: stratified sample (better for ML)
# Sample proportionally by climate zone
sample = meta_ca.groupby('in.building_america_climate_zone', 
                          group_keys=False).apply(
    lambda x: x.sample(min(len(x), 200))
)
print(f"Sample size: {len(sample)}")
# Get the building IDs
building_ids = sample.index.tolist()

Sample size: 800


C:\Users\moham\AppData\Local\Temp\ipykernel_15692\3270520287.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_keys=False).apply(


In [3]:
import os

timeseries_path = f"{base_path}/timeseries_individual_buildings/by_state/upgrade=0/state=CA"

# 1. List all files in the CA upgrade=0 folder
print("Fetching list of available files from S3...")
all_files = fs.ls(timeseries_path)

# 2. Extract the building ID from the filenames
# Example: '.../10002-0.parquet' -> 10002
available_ids = []
for f in all_files:
    filename = f.split('/')[-1]
    if filename.endswith('-0.parquet'):
        b_id = int(filename.replace('-0.parquet', ''))
        available_ids.append(b_id)

print(f"Found {len(available_ids)} buildings with timeseries data in this folder.")

Fetching list of available files from S3...
Found 57075 buildings with timeseries data in this folder.


In [4]:
# Filter metadata to only include buildings that exist in the timeseries folder
meta_available = meta_ca[meta_ca.index.isin(available_ids)]
# Now perform your stratified sample on this filtered set
sample = meta_available.groupby('in.building_america_climate_zone', 
                                group_keys=False).apply(
    lambda x: x.sample(min(len(x), 200))
)
building_ids = sample.index.tolist()
print(f"New sample size: {len(building_ids)}")

New sample size: 544


C:\Users\moham\AppData\Local\Temp\ipykernel_15692\3368196649.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_keys=False).apply(


In [5]:
import os
import pandas as pd
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

def load_building(building_id, fs):
    """Load timeseries for a single building."""
    try:
        path = f"s3://{timeseries_path}/{building_id}-0.parquet"
        df = pd.read_parquet(path, filesystem=fs)
        df['building_id'] = building_id
        return df
    except Exception as e:
        print(f"Failed to load {building_id}: {e}")
        return None

building_ids_sample = building_ids[:200]
batches = [building_ids_sample[i:i+20] for i in range(0, len(building_ids_sample), 20)]

output_file = "ca_timeseries.parquet"
writer = None

for batch_num, batch in enumerate(tqdm(batches, desc="Batches")):
    batch_dfs = []
    for bid in tqdm(batch, desc=f"Batch {batch_num}", leave=False):
        df = load_building(bid, fs)
        if df is not None:
            batch_dfs.append(df)
    
    if batch_dfs:
        batch_df = pd.concat(batch_dfs, ignore_index=True)
        table = pa.Table.from_pandas(batch_df)
        
        if writer is None:
            writer = pq.ParquetWriter(output_file, table.schema)
        
        writer.write_table(table)
        del batch_df, batch_dfs

if writer:
    writer.close()

print(f"Done! Saved to {output_file}")

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 10/10 [07:22<00:00, 44.29s/it]

Done! Saved to ca_timeseries.parquet


In [6]:
weather_path = f"{base_path}/weather/state=CA"

def load_weather(weather_id, fs):
    """Load weather file using the G-prefixed ID from the screenshot."""
    try:
        # Construct the path using the ID (e.g., G0600010)
        # Note: We do NOT add 's3://' here because it's already in the read_csv call
        path = f"s3://{weather_path}/{weather_id}_2018.csv"
        
        # Use storage_options to pass the filesystem configuration
        df = pd.read_csv(path, 
                         storage_options={'anon': True},
                         parse_dates=['date_time'],
                         index_col='date_time')
        return df
    except Exception as e:
        print(f"Failed to load weather {weather_id}: {e}")
        return None

# 1. Identify the column containing the G-prefixed IDs. 
# In the 2025 release, this is typically 'in.county' or 'in.weather_file_city'.
# Let's verify which one it is:
print("Sample IDs from metadata:")
print(sample['in.county'].head()) 

# 2. Get unique weather IDs from your sample
weather_ids = sample['in.county'].unique()

# 3. Load the files
weather_dict = {}
for wid in tqdm(weather_ids):
    weather_dict[wid] = load_weather(wid, fs)

Sample IDs from metadata:
20445    G0600570
15737    G0600490
49088    G0600030
46777    G0600570
38987    G0600570
Name: in.county, dtype: object


100%|██████████████████████████████████████████████████████████████████████████████████| 52/52 [00:31<00:00,  1.63it/s]


In [7]:
def build_training_dataset_vectorized(timeseries, meta, weather_dict):
    # 1. Clean timestamps
    timeseries['timestamp'] = pd.to_datetime(timeseries['timestamp'])
    
    # 2. Map building_id to weather_id (County ID)
    # Check if column is 'in.county' or just 'county'
    county_col = 'in.county' if 'in.county' in meta.columns else 'county'
    weather_mapping = meta[county_col].to_dict()
    timeseries['weather_id'] = timeseries['building_id'].map(weather_mapping)
    
    # 3. Create Multi-Indexed weather DF
    all_weather_list = []
    for wid, df in weather_dict.items():
        if df is not None:
            temp_df = df.copy()
            temp_df.index = pd.to_datetime(temp_df.index)
            temp_df['weather_id'] = wid
            all_weather_list.append(temp_df)
            
    all_weather = pd.concat(all_weather_list).set_index(['weather_id'], append=True)
    
    # 4. Merge Timeseries + Weather
    df = timeseries.merge(
        all_weather, 
        left_on=['timestamp', 'weather_id'], 
        right_index=True,
        how='inner'
    )
    
    # 5. Merge Metadata (Handle potential naming differences)
    # Define the core features we want, checking for 'in.' prefix dynamically
    core_features = ['sqft', 'vintage', 'hvac_cooling_type', 'bedrooms']
    available_meta_cols = []
    
    for feat in core_features:
        if f"in.{feat}" in meta.columns:
            available_meta_cols.append(f"in.{feat}")
        elif feat in meta.columns:
            available_meta_cols.append(feat)
            
    df = df.merge(meta[available_meta_cols], left_on='building_id', right_index=True)
    
    # 6. Time Features
    df['hour'] = df['timestamp'].dt.hour
    df['month'] = df['timestamp'].dt.month
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'] >= 5
    
    return df

In [8]:
timeseries = pd.read_parquet("ca_timeseries.parquet")
data = build_training_dataset_vectorized(timeseries, meta_ca, weather_dict)

In [9]:
data

,bldg_id,timestamp,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,...,Global Horizontal Radiation [W/m2],Direct Normal Radiation [W/m2],Diffuse Horizontal Radiation [W/m2],in.vintage,in.hvac_cooling_type,in.bedrooms,hour,month,day_of_week,is_weekend
3,20445,2018-01-01 01:00:00,2179.0,0.0,0.0000,0.04505,0.01250,0.00000,0.0,0.0,...,0.0,0.0,0.0,1980s,Room AC,2,1,1,0,False
7,20445,2018-01-01 02:00:00,2179.0,0.0,1.2310,0.00000,0.01250,0.00000,0.0,0.0,...,0.0,0.0,0.0,1980s,Room AC,2,2,1,0,False
11,20445,2018-01-01 03:00:00,2179.0,0.0,0.0000,0.00000,0.01250,0.00000,0.0,0.0,...,0.0,0.0,0.0,1980s,Room AC,2,3,1,0,False
15,20445,2018-01-01 04:00:00,2179.0,0.0,0.7386,0.03859,0.00000,0.00000,0.0,0.0,...,0.0,0.0,0.0,1980s,Room AC,2,4,1,0,False
19,20445,2018-01-01 05:00:00,2179.0,0.0,0.0000,0.00000,0.00000,0.00000,0.0,0.0,...,0.0,0.0,0.0,1980s,Room AC,2,5,1,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007983,9685,2018-12-31 20:00:00,1228.0,0.0,0.0000,0.00000,0.00565,0.00147,0.0,0.0,...,0.0,0.0,0.0,1990s,Room AC,1,20,12,0,False
7007987,9685,2018-12-31 21:00:00,1228.0,0.0,0.0000,0.00000,0.00000,0.00000,0.0,0.0,...,0.0,0.0,0.0,1990s,Room AC,1,21,12,0,False
7007991,9685,2018-12-31 22:00:00,1228.0,0.0,0.0000,0.00000,0.00000,0.00000,0.0,0.0,...,0.0,0.0,0.0,1990s,Room AC,1,22,12,0,False
7007995,9685,2018-12-31 23:00:00,1228.0,0.0,0.0000,0.00000,0.00000,0.00000,0.0,0.0,...,0.0,0.0,0.0,1990s,Room AC,1,23,12,0,False


In [10]:
data.to_parquet('buildings_data.parquet')

In [14]:
data_small = data.head(1000)

In [15]:
data_small.to_parquet('small.parquet')

In [3]:
import pandas as pd

data = pd.read_parquet('buildings_data.parquet')

COLUMN_RENAME_MAP = {
    # Identifiers & Time
    'bldg_id': 'building_id',
    'timestamp': 'timestamp',
    'hour': 'hour',
    'month': 'month',
    'day_of_week': 'day_of_week',
    'is_weekend': 'is_weekend',
    'state': 'state',
    'weather_id': 'weather_id',
    'upgrade': 'upgrade',

    # Building Metadata
    'in.sqft': 'sqft',
    'in.vintage': 'vintage',
    'in.hvac_cooling_type': 'hvac_cooling_type',
    'in.bedrooms': 'bedrooms',

    # Electricity consumption (kWh)
    'out.electricity.ceiling_fan.energy_consumption..kwh': 'elec_ceiling_fan_kwh',
    'out.electricity.clothes_dryer.energy_consumption..kwh': 'elec_clothes_dryer_kwh',
    'out.electricity.clothes_washer.energy_consumption..kwh': 'elec_clothes_washer_kwh',
    'out.electricity.cooling.energy_consumption..kwh': 'elec_cooling_kwh',
    'out.electricity.cooling_fans_pumps.energy_consumption..kwh': 'elec_cooling_fans_pumps_kwh',
    'out.electricity.dishwasher.energy_consumption..kwh': 'elec_dishwasher_kwh',
    'out.electricity.ev_charging.energy_consumption..kwh': 'elec_ev_charging_kwh',
    'out.electricity.freezer.energy_consumption..kwh': 'elec_freezer_kwh',
    'out.electricity.heating.energy_consumption..kwh': 'elec_heating_kwh',
    'out.electricity.heating_fans_pumps.energy_consumption..kwh': 'elec_heating_fans_pumps_kwh',
    'out.electricity.heating_hp_bkup.energy_consumption..kwh': 'elec_heating_hp_backup_kwh',
    'out.electricity.heating_hp_bkup_fa.energy_consumption..kwh': 'elec_heating_hp_backup_fa_kwh',
    'out.electricity.hot_water.energy_consumption..kwh': 'elec_hot_water_kwh',
    'out.electricity.hot_water_solar_th.energy_consumption..kwh': 'elec_hot_water_solar_thermal_kwh',
    'out.electricity.lighting_exterior.energy_consumption..kwh': 'elec_lighting_exterior_kwh',
    'out.electricity.lighting_garage.energy_consumption..kwh': 'elec_lighting_garage_kwh',
    'out.electricity.lighting_interior.energy_consumption..kwh': 'elec_lighting_interior_kwh',
    'out.electricity.mech_vent.energy_consumption..kwh': 'elec_mechanical_ventilation_kwh',
    'out.electricity.net.energy_consumption..kwh': 'elec_net_kwh',
    'out.electricity.permanent_spa_heat.energy_consumption..kwh': 'elec_spa_heating_kwh',
    'out.electricity.permanent_spa_pump.energy_consumption..kwh': 'elec_spa_pump_kwh',
    'out.electricity.plug_loads.energy_consumption..kwh': 'elec_plug_loads_kwh',
    'out.electricity.pool_heater.energy_consumption..kwh': 'elec_pool_heater_kwh',
    'out.electricity.pool_pump.energy_consumption..kwh': 'elec_pool_pump_kwh',
    'out.electricity.pv.energy_consumption..kwh': 'elec_solar_pv_kwh',
    'out.electricity.range_oven.energy_consumption..kwh': 'elec_range_oven_kwh',
    'out.electricity.refrigerator.energy_consumption..kwh': 'elec_refrigerator_kwh',
    'out.electricity.television.energy_consumption..kwh': 'elec_television_kwh',
    'out.electricity.total.energy_consumption..kwh': 'elec_total_kwh',
    'out.electricity.well_pump.energy_consumption..kwh': 'elec_well_pump_kwh',

    # Electricity intensity (kWh/ft2)
    'out.electricity.ceiling_fan.energy_consumption_intensity..kwh_per_ft2': 'elec_ceiling_fan_kwh_per_ft2',
    'out.electricity.clothes_dryer.energy_consumption_intensity..kwh_per_ft2': 'elec_clothes_dryer_kwh_per_ft2',
    'out.electricity.clothes_washer.energy_consumption_intensity..kwh_per_ft2': 'elec_clothes_washer_kwh_per_ft2',
    'out.electricity.cooling.energy_consumption_intensity..kwh_per_ft2': 'elec_cooling_kwh_per_ft2',
    'out.electricity.cooling_fans_pumps.energy_consumption_intensity..kwh_per_ft2': 'elec_cooling_fans_pumps_kwh_per_ft2',
    'out.electricity.dishwasher.energy_consumption_intensity..kwh_per_ft2': 'elec_dishwasher_kwh_per_ft2',
    'out.electricity.ev_charging.energy_consumption_intensity..kwh_per_ft2': 'elec_ev_charging_kwh_per_ft2',
    'out.electricity.freezer.energy_consumption_intensity..kwh_per_ft2': 'elec_freezer_kwh_per_ft2',
    'out.electricity.heating.energy_consumption_intensity..kwh_per_ft2': 'elec_heating_kwh_per_ft2',
    'out.electricity.heating_fans_pumps.energy_consumption_intensity..kwh_per_ft2': 'elec_heating_fans_pumps_kwh_per_ft2',
    'out.electricity.heating_hp_bkup.energy_consumption_intensity..kwh_per_ft2': 'elec_heating_hp_backup_kwh_per_ft2',
    'out.electricity.heating_hp_bkup_fa.energy_consumption_intensity..kwh_per_ft2': 'elec_heating_hp_backup_fa_kwh_per_ft2',
    'out.electricity.hot_water.energy_consumption_intensity..kwh_per_ft2': 'elec_hot_water_kwh_per_ft2',
    'out.electricity.hot_water_solar_th.energy_consumption_intensity..kwh_per_ft2': 'elec_hot_water_solar_thermal_kwh_per_ft2',
    'out.electricity.lighting_exterior.energy_consumption_intensity..kwh_per_ft2': 'elec_lighting_exterior_kwh_per_ft2',
    'out.electricity.lighting_garage.energy_consumption_intensity..kwh_per_ft2': 'elec_lighting_garage_kwh_per_ft2',
    'out.electricity.lighting_interior.energy_consumption_intensity..kwh_per_ft2': 'elec_lighting_interior_kwh_per_ft2',
    'out.electricity.mech_vent.energy_consumption_intensity..kwh_per_ft2': 'elec_mechanical_ventilation_kwh_per_ft2',
    'out.electricity.net.energy_consumption_intensity..kwh_per_ft2': 'elec_net_kwh_per_ft2',
    'out.electricity.permanent_spa_heat.energy_consumption_intensity..kwh_per_ft2': 'elec_spa_heating_kwh_per_ft2',
    'out.electricity.permanent_spa_pump.energy_consumption_intensity..kwh_per_ft2': 'elec_spa_pump_kwh_per_ft2',
    'out.electricity.plug_loads.energy_consumption_intensity..kwh_per_ft2': 'elec_plug_loads_kwh_per_ft2',
    'out.electricity.pool_heater.energy_consumption_intensity..kwh_per_ft2': 'elec_pool_heater_kwh_per_ft2',
    'out.electricity.pool_pump.energy_consumption_intensity..kwh_per_ft2': 'elec_pool_pump_kwh_per_ft2',
    'out.electricity.pv.energy_consumption_intensity..kwh_per_ft2': 'elec_solar_pv_kwh_per_ft2',
    'out.electricity.range_oven.energy_consumption_intensity..kwh_per_ft2': 'elec_range_oven_kwh_per_ft2',
    'out.electricity.refrigerator.energy_consumption_intensity..kwh_per_ft2': 'elec_refrigerator_kwh_per_ft2',
    'out.electricity.television.energy_consumption_intensity..kwh_per_ft2': 'elec_television_kwh_per_ft2',
    'out.electricity.total.energy_consumption_intensity..kwh_per_ft2': 'elec_total_kwh_per_ft2',
    'out.electricity.well_pump.energy_consumption_intensity..kwh_per_ft2': 'elec_well_pump_kwh_per_ft2',

    # Natural gas (kWh)
    'out.natural_gas.clothes_dryer.energy_consumption..kwh': 'gas_clothes_dryer_kwh',
    'out.natural_gas.fireplace.energy_consumption..kwh': 'gas_fireplace_kwh',
    'out.natural_gas.grill.energy_consumption..kwh': 'gas_grill_kwh',
    'out.natural_gas.heating.energy_consumption..kwh': 'gas_heating_kwh',
    'out.natural_gas.heating_hp_bkup.energy_consumption..kwh': 'gas_heating_hp_backup_kwh',
    'out.natural_gas.hot_water.energy_consumption..kwh': 'gas_hot_water_kwh',
    'out.natural_gas.lighting.energy_consumption..kwh': 'gas_lighting_kwh',
    'out.natural_gas.permanent_spa_heat.energy_consumption..kwh': 'gas_spa_heating_kwh',
    'out.natural_gas.pool_heater.energy_consumption..kwh': 'gas_pool_heater_kwh',
    'out.natural_gas.range_oven.energy_consumption..kwh': 'gas_range_oven_kwh',
    'out.natural_gas.total.energy_consumption..kwh': 'gas_total_kwh',

    # Natural gas intensity (kWh/ft2)
    'out.natural_gas.clothes_dryer.energy_consumption_intensity..kwh_per_ft2': 'gas_clothes_dryer_kwh_per_ft2',
    'out.natural_gas.fireplace.energy_consumption_intensity..kwh_per_ft2': 'gas_fireplace_kwh_per_ft2',
    'out.natural_gas.grill.energy_consumption_intensity..kwh_per_ft2': 'gas_grill_kwh_per_ft2',
    'out.natural_gas.heating.energy_consumption_intensity..kwh_per_ft2': 'gas_heating_kwh_per_ft2',
    'out.natural_gas.heating_hp_bkup.energy_consumption_intensity..kwh_per_ft2': 'gas_heating_hp_backup_kwh_per_ft2',
    'out.natural_gas.hot_water.energy_consumption_intensity..kwh_per_ft2': 'gas_hot_water_kwh_per_ft2',
    'out.natural_gas.lighting.energy_consumption_intensity..kwh_per_ft2': 'gas_lighting_kwh_per_ft2',
    'out.natural_gas.permanent_spa_heat.energy_consumption_intensity..kwh_per_ft2': 'gas_spa_heating_kwh_per_ft2',
    'out.natural_gas.pool_heater.energy_consumption_intensity..kwh_per_ft2': 'gas_pool_heater_kwh_per_ft2',
    'out.natural_gas.range_oven.energy_consumption_intensity..kwh_per_ft2': 'gas_range_oven_kwh_per_ft2',
    'out.natural_gas.total.energy_consumption_intensity..kwh_per_ft2': 'gas_total_kwh_per_ft2',

    # Propane (kWh)
    'out.propane.clothes_dryer.energy_consumption..kwh': 'propane_clothes_dryer_kwh',
    'out.propane.heating.energy_consumption..kwh': 'propane_heating_kwh',
    'out.propane.hot_water.energy_consumption..kwh': 'propane_hot_water_kwh',
    'out.propane.range_oven.energy_consumption..kwh': 'propane_range_oven_kwh',
    'out.propane.total.energy_consumption..kwh': 'propane_total_kwh',

    # Propane intensity (kWh/ft2)
    'out.propane.clothes_dryer.energy_consumption_intensity..kwh_per_ft2': 'propane_clothes_dryer_kwh_per_ft2',
    'out.propane.heating.energy_consumption_intensity..kwh_per_ft2': 'propane_heating_kwh_per_ft2',
    'out.propane.hot_water.energy_consumption_intensity..kwh_per_ft2': 'propane_hot_water_kwh_per_ft2',
    'out.propane.range_oven.energy_consumption_intensity..kwh_per_ft2': 'propane_range_oven_kwh_per_ft2',
    'out.propane.total.energy_consumption_intensity..kwh_per_ft2': 'propane_total_kwh_per_ft2',

    # Fuel oil (kWh)
    'out.fuel_oil.heating.energy_consumption..kwh': 'fuel_oil_heating_kwh',
    'out.fuel_oil.hot_water.energy_consumption..kwh': 'fuel_oil_hot_water_kwh',
    'out.fuel_oil.total.energy_consumption..kwh': 'fuel_oil_total_kwh',

    # Fuel oil intensity (kWh/ft2)
    'out.fuel_oil.heating.energy_consumption_intensity..kwh_per_ft2': 'fuel_oil_heating_kwh_per_ft2',
    'out.fuel_oil.hot_water.energy_consumption_intensity..kwh_per_ft2': 'fuel_oil_hot_water_kwh_per_ft2',
    'out.fuel_oil.total.energy_consumption_intensity..kwh_per_ft2': 'fuel_oil_total_kwh_per_ft2',

    # Site energy totals
    'out.site_energy.net.energy_consumption..kwh': 'site_energy_net_kwh',
    'out.site_energy.total.energy_consumption..kwh': 'site_energy_total_kwh',
    'out.site_energy.net.energy_consumption_intensity..kwh_per_ft2': 'site_energy_net_kwh_per_ft2',
    'out.site_energy.total.energy_consumption_intensity..kwh_per_ft2': 'site_energy_total_kwh_per_ft2',

    # Emissions - electricity (kg CO2e)
    'out.emissions.electricity.total.aer_highrecost_avg..co2e_kg': 'emissions_elec_aer_high_re_cost_kg',
    'out.emissions.electricity.total.aer_highrecostlowngprice_avg..co2e_kg': 'emissions_elec_aer_high_re_low_ng_kg',
    'out.emissions.electricity.total.aer_lowrecost_avg..co2e_kg': 'emissions_elec_aer_low_re_cost_kg',
    'out.emissions.electricity.total.aer_lowrecosthighngprice_avg..co2e_kg': 'emissions_elec_aer_low_re_high_ng_kg',
    'out.emissions.electricity.total.aer_midcase_avg..co2e_kg': 'emissions_elec_aer_midcase_kg',
    'out.emissions.electricity.total.lrmer_highrecost_15..co2e_kg': 'emissions_elec_lrmer_high_re_15yr_kg',
    'out.emissions.electricity.total.lrmer_highrecost_25..co2e_kg': 'emissions_elec_lrmer_high_re_25yr_kg',
    'out.emissions.electricity.total.lrmer_highrecostlowngprice_15..co2e_kg': 'emissions_elec_lrmer_high_re_low_ng_15yr_kg',
    'out.emissions.electricity.total.lrmer_highrecostlowngprice_25..co2e_kg': 'emissions_elec_lrmer_high_re_low_ng_25yr_kg',
    'out.emissions.electricity.total.lrmer_lowrecost_15..co2e_kg': 'emissions_elec_lrmer_low_re_15yr_kg',
    'out.emissions.electricity.total.lrmer_lowrecost_25..co2e_kg': 'emissions_elec_lrmer_low_re_25yr_kg',
    'out.emissions.electricity.total.lrmer_lowrecosthighngprice_15..co2e_kg': 'emissions_elec_lrmer_low_re_high_ng_15yr_kg',
    'out.emissions.electricity.total.lrmer_lowrecosthighngprice_25..co2e_kg': 'emissions_elec_lrmer_low_re_high_ng_25yr_kg',
    'out.emissions.electricity.total.lrmer_midcase_15..co2e_kg': 'emissions_elec_lrmer_midcase_15yr_kg',
    'out.emissions.electricity.total.lrmer_midcase_25..co2e_kg': 'emissions_elec_lrmer_midcase_25yr_kg',

    # Emissions - other fuels (kg CO2e)
    'out.emissions.natural_gas.total..co2e_kg': 'emissions_gas_total_kg',
    'out.emissions.propane.total..co2e_kg': 'emissions_propane_total_kg',
    'out.emissions.fuel_oil.total..co2e_kg': 'emissions_fuel_oil_total_kg',

    # Emissions - total (kg CO2e)
    'out.emissions.total.aer_highrecost_avg..co2e_kg': 'emissions_total_aer_high_re_cost_kg',
    'out.emissions.total.aer_highrecostlowngprice_avg..co2e_kg': 'emissions_total_aer_high_re_low_ng_kg',
    'out.emissions.total.aer_lowrecost_avg..co2e_kg': 'emissions_total_aer_low_re_cost_kg',
    'out.emissions.total.aer_lowrecosthighngprice_avg..co2e_kg': 'emissions_total_aer_low_re_high_ng_kg',
    'out.emissions.total.aer_midcase_avg..co2e_kg': 'emissions_total_aer_midcase_kg',
    'out.emissions.total.lrmer_highrecost_15..co2e_kg': 'emissions_total_lrmer_high_re_15yr_kg',
    'out.emissions.total.lrmer_highrecost_25..co2e_kg': 'emissions_total_lrmer_high_re_25yr_kg',
    'out.emissions.total.lrmer_highrecostlowngprice_15..co2e_kg': 'emissions_total_lrmer_high_re_low_ng_15yr_kg',
    'out.emissions.total.lrmer_highrecostlowngprice_25..co2e_kg': 'emissions_total_lrmer_high_re_low_ng_25yr_kg',
    'out.emissions.total.lrmer_lowrecost_15..co2e_kg': 'emissions_total_lrmer_low_re_15yr_kg',
    'out.emissions.total.lrmer_lowrecost_25..co2e_kg': 'emissions_total_lrmer_low_re_25yr_kg',
    'out.emissions.total.lrmer_lowrecosthighngprice_15..co2e_kg': 'emissions_total_lrmer_low_re_high_ng_15yr_kg',
    'out.emissions.total.lrmer_lowrecosthighngprice_25..co2e_kg': 'emissions_total_lrmer_low_re_high_ng_25yr_kg',
    'out.emissions.total.lrmer_midcase_15..co2e_kg': 'emissions_total_lrmer_midcase_15yr_kg',
    'out.emissions.total.lrmer_midcase_25..co2e_kg': 'emissions_total_lrmer_midcase_25yr_kg',

    # Unmet hours
    'out.unmet_hours.cooling..hour': 'unmet_hours_cooling',
    'out.unmet_hours.ev_driving..hour': 'unmet_hours_ev_driving',
    'out.unmet_hours.heating..hour': 'unmet_hours_heating',

    # Hot water (gallons)
    'out.hot_water.clothes_washer..gal': 'hot_water_clothes_washer_gal',
    'out.hot_water.dishwasher..gal': 'hot_water_dishwasher_gal',
    'out.hot_water.distribution_waste..gal': 'hot_water_distribution_waste_gal',
    'out.hot_water.fixtures..gal': 'hot_water_fixtures_gal',

    # Thermal loads (kBtu)
    'out.load.cooling.energy_delivered..kbtu': 'load_cooling_kbtu',
    'out.load.heating.energy_delivered..kbtu': 'load_heating_kbtu',
    'out.load.heating_heat_pump_backup..kbtu': 'load_heating_hp_backup_kbtu',
    'out.load.hot_water.energy_delivered..kbtu': 'load_hot_water_kbtu',
    'out.load.hot_water_solar_thermal..kbtu': 'load_hot_water_solar_thermal_kbtu',
    'out.load.hot_water_tank_losses..kbtu': 'load_hot_water_tank_losses_kbtu',

    # Indoor airflow (CFM)
    'out.indoor_airflow.infiltration..cfm': 'airflow_infiltration_cfm',
    'out.indoor_airflow.mechanical_ventilation..cfm': 'airflow_mechanical_ventilation_cfm',
    'out.indoor_airflow.natural_ventilation..cfm': 'airflow_natural_ventilation_cfm',

    # Indoor environment
    'out.indoor_dewpoint_temperature.conditioned_space..c': 'indoor_dewpoint_temp_c',
    'out.indoor_humidity_ratio.conditioned_space..kgwater_per_kgdryair': 'indoor_humidity_ratio',
    'out.indoor_operative_temperature.conditioned_space..c': 'indoor_operative_temp_c',
    'out.indoor_radiant_temperature.conditioned_space..c': 'indoor_radiant_temp_c',
    'out.indoor_relative_humidity.conditioned_space..percentage': 'indoor_relative_humidity_pct',
    'out.indoor_temperature.conditioned_space..c': 'indoor_temp_c',

    # Outdoor environment
    'out.outdoor_air_drybulb_temp..c': 'outdoor_drybulb_temp_c',
    'out.outdoor_air_relative_humidity..percentage': 'outdoor_relative_humidity_pct',
    'out.outdoor_air_wetbulb_temp..c': 'outdoor_wetbulb_temp_c',
    'out.outdoor_humidity_ratio..kgwater_per_kgdryair': 'outdoor_humidity_ratio',

    # People / occupancy
    'out.people.sensible_heating_rate..watt': 'people_sensible_heat_w',
    'out.people.total_heating_rate..watt': 'people_total_heat_w',

    # Solar / weather outputs
    'out.weather.diffuse_solar_radiation..watt_per_m2': 'diffuse_solar_radiation_w_m2',
    'out.weather.direct_normal_solar_radiation..watt_per_m2': 'direct_normal_radiation_w_m2',
    'out.weather.wind_speed..meter_per_second': 'wind_speed_m_s',

    # Schedules (0-1 fractions)
    'out.schedules.ceiling_fan': 'schedule_ceiling_fan',
    'out.schedules.clothes_dryer': 'schedule_clothes_dryer',
    'out.schedules.clothes_washer': 'schedule_clothes_washer',
    'out.schedules.cooking_range': 'schedule_cooking_range',
    'out.schedules.cooling_setpoint..c': 'cooling_setpoint_c',
    'out.schedules.dishwasher': 'schedule_dishwasher',
    'out.schedules.electric_vehicle_charging': 'schedule_ev_charging',
    'out.schedules.electric_vehicle_discharging': 'schedule_ev_discharging',
    'out.schedules.heating_setpoint..c': 'heating_setpoint_c',
    'out.schedules.hot_water_clothes_washer': 'schedule_hot_water_clothes_washer',
    'out.schedules.hot_water_dishwasher': 'schedule_hot_water_dishwasher',
    'out.schedules.hot_water_fixtures': 'schedule_hot_water_fixtures',
    'out.schedules.lighting_garage': 'schedule_lighting_garage',
    'out.schedules.lighting_interior': 'schedule_lighting_interior',
    'out.schedules.no_space_cooling': 'schedule_no_cooling',
    'out.schedules.no_space_heating': 'schedule_no_heating',
    'out.schedules.occupants': 'schedule_occupants',
    'out.schedules.peak_period': 'schedule_peak_period',
    'out.schedules.plug_loads_other': 'schedule_plug_loads_other',
    'out.schedules.plug_loads_tv': 'schedule_plug_loads_tv',
    'out.schedules.power_outage': 'schedule_power_outage',
    'out.schedules.pre_peak_period': 'schedule_pre_peak_period',
    'out.schedules.vacancy': 'schedule_vacancy',

    # Weather file columns (from merge)
    'Dry Bulb Temperature [°C]': 'weather_drybulb_temp_c',
    'Relative Humidity [%]': 'weather_relative_humidity_pct',
    'Wind Speed [m/s]': 'weather_wind_speed_m_s',
    'Wind Direction [Deg]': 'weather_wind_direction_deg',
    'Global Horizontal Radiation [W/m2]': 'weather_global_horizontal_radiation_w_m2',
    'Direct Normal Radiation [W/m2]': 'weather_direct_normal_radiation_w_m2',
    'Diffuse Horizontal Radiation [W/m2]': 'weather_diffuse_horizontal_radiation_w_m2',
}


def rename_columns(df):
    """Rename all columns to readable names. Columns not in the map are left unchanged."""
    return df.rename(columns=COLUMN_RENAME_MAP)


data = rename_columns(data)
data = data.loc[:, ~data.columns.duplicated()]
data.to_parquet('buildings_data_renamed.parquet')

In [19]:
data

,building_id,timestamp,sqft,elec_ceiling_fan_kwh,elec_clothes_dryer_kwh,elec_clothes_washer_kwh,elec_cooling_kwh,elec_cooling_fans_pumps_kwh,elec_dishwasher_kwh,elec_ev_charging_kwh,...,weather_global_horizontal_radiation_w_m2,weather_direct_normal_radiation_w_m2,weather_diffuse_horizontal_radiation_w_m2,vintage,hvac_cooling_type,bedrooms,hour,month,day_of_week,is_weekend
3,56292,2018-01-01 01:00:00,1698.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,<1940,Room AC,4,1,1,0,False
7,56292,2018-01-01 02:00:00,1698.0,0.00217,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,<1940,Room AC,4,2,1,0,False
11,56292,2018-01-01 03:00:00,1698.0,0.00217,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,<1940,Room AC,4,3,1,0,False
15,56292,2018-01-01 04:00:00,1698.0,0.00217,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,<1940,Room AC,4,4,1,0,False
19,56292,2018-01-01 05:00:00,1698.0,0.00217,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,<1940,Room AC,4,5,1,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3503983,24571,2018-12-31 20:00:00,1682.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1980s,None,2,20,12,0,False
3503987,24571,2018-12-31 21:00:00,1682.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1980s,None,2,21,12,0,False
3503991,24571,2018-12-31 22:00:00,1682.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1980s,None,2,22,12,0,False
3503995,24571,2018-12-31 23:00:00,1682.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1980s,None,2,23,12,0,False


In [21]:
for col in data.columns:
    print(col)

building_id
timestamp
sqft
elec_ceiling_fan_kwh
elec_clothes_dryer_kwh
elec_clothes_washer_kwh
elec_cooling_kwh
elec_cooling_fans_pumps_kwh
elec_dishwasher_kwh
elec_ev_charging_kwh
elec_freezer_kwh
elec_heating_kwh
elec_heating_fans_pumps_kwh
elec_heating_hp_backup_kwh
elec_heating_hp_backup_fa_kwh
elec_hot_water_kwh
elec_hot_water_solar_thermal_kwh
elec_lighting_exterior_kwh
elec_lighting_garage_kwh
elec_lighting_interior_kwh
elec_mechanical_ventilation_kwh
elec_net_kwh
elec_spa_heating_kwh
elec_spa_pump_kwh
elec_plug_loads_kwh
elec_pool_heater_kwh
elec_pool_pump_kwh
elec_solar_pv_kwh
elec_range_oven_kwh
elec_refrigerator_kwh
elec_television_kwh
elec_total_kwh
elec_well_pump_kwh
fuel_oil_heating_kwh
fuel_oil_hot_water_kwh
fuel_oil_total_kwh
gas_clothes_dryer_kwh
gas_fireplace_kwh
gas_grill_kwh
gas_heating_kwh
gas_heating_hp_backup_kwh
gas_hot_water_kwh
gas_lighting_kwh
gas_spa_heating_kwh
gas_pool_heater_kwh
gas_range_oven_kwh
gas_total_kwh
propane_clothes_dryer_kwh
propane_heating_kw

In [4]:
# Define the prefixes or names of columns you MUST keep
keep_prefixes = ['building_id', 'elec_', 'weather_', 'timestamp', 'hour', 'month', 'day', 'sqft', 'bedrooms', 'hvac_cooling_type', 'is_weekend']

# Create a list of all columns that match those prefixes
cols_to_keep = [col for col in data.columns if any(col.startswith(p) for p in keep_prefixes)]

# Filter the dataframe
data_elec_only = data[cols_to_keep]

print(f"Removed {len(data.columns) - len(cols_to_keep)} non-electric columns.")

Removed 132 non-electric columns.


In [25]:
for col in data_elec_only.columns:
    print(col)

timestamp
sqft
elec_ceiling_fan_kwh
elec_clothes_dryer_kwh
elec_clothes_washer_kwh
elec_cooling_kwh
elec_cooling_fans_pumps_kwh
elec_dishwasher_kwh
elec_ev_charging_kwh
elec_freezer_kwh
elec_heating_kwh
elec_heating_fans_pumps_kwh
elec_heating_hp_backup_kwh
elec_heating_hp_backup_fa_kwh
elec_hot_water_kwh
elec_hot_water_solar_thermal_kwh
elec_lighting_exterior_kwh
elec_lighting_garage_kwh
elec_lighting_interior_kwh
elec_mechanical_ventilation_kwh
elec_net_kwh
elec_spa_heating_kwh
elec_spa_pump_kwh
elec_plug_loads_kwh
elec_pool_heater_kwh
elec_pool_pump_kwh
elec_solar_pv_kwh
elec_range_oven_kwh
elec_refrigerator_kwh
elec_television_kwh
elec_total_kwh
elec_well_pump_kwh
elec_ceiling_fan_kwh_per_ft2
elec_clothes_dryer_kwh_per_ft2
elec_clothes_washer_kwh_per_ft2
elec_cooling_kwh_per_ft2
elec_cooling_fans_pumps_kwh_per_ft2
elec_dishwasher_kwh_per_ft2
elec_ev_charging_kwh_per_ft2
elec_freezer_kwh_per_ft2
elec_heating_kwh_per_ft2
elec_heating_fans_pumps_kwh_per_ft2
elec_heating_hp_backup_kwh_

In [5]:
intensity_cols = [c for c in data_elec_only.columns if '_per_ft2' in c]
data_final = data_elec_only.drop(columns=intensity_cols)

hp_cols = [col for col in data_final.columns if 'hp_backup' in col]
data_final = data_final.drop(columns=hp_cols)

data_final = data_final.drop(columns=['weather_id', 'timestamp', 'elec_spa_heating_kwh', 'elec_spa_pump_kwh', 'elec_cooling_fans_pumps_kwh', 'elec_heating_fans_pumps_kwh', 'elec_hot_water_solar_thermal_kwh', 'elec_well_pump_kwh','elec_clothes_dryer_kwh', 'elec_dishwasher_kwh', 'elec_total_kwh', 'elec_solar_pv_kwh', 'elec_net_kwh', 'elec_ev_charging_kwh', 'elec_pool_heater_kwh', 'elec_pool_pump_kwh', 'elec_lighting_garage_kwh', 'elec_mechanical_ventilation_kwh', ])

In [8]:
data_final

,building_id,sqft,elec_ceiling_fan_kwh,elec_clothes_washer_kwh,elec_cooling_kwh,elec_freezer_kwh,elec_heating_kwh,elec_hot_water_kwh,elec_lighting_exterior_kwh,elec_lighting_interior_kwh,...,weather_wind_direction_deg,weather_global_horizontal_radiation_w_m2,weather_direct_normal_radiation_w_m2,weather_diffuse_horizontal_radiation_w_m2,hvac_cooling_type,bedrooms,hour,month,day_of_week,is_weekend
3,20445,2179.0,0.0,0.04505,0.01250,0.02611,0.15115,0.0,0.00186,0.05852,...,0.0,0.0,0.0,0.0,Room AC,2,1,1,0,False
7,20445,2179.0,0.0,0.00000,0.01250,0.02497,1.35371,0.0,0.00160,0.05071,...,0.0,0.0,0.0,0.0,Room AC,2,2,1,0,False
11,20445,2179.0,0.0,0.00000,0.01250,0.02327,1.08371,0.0,0.00135,0.01465,...,0.0,0.0,0.0,0.0,Room AC,2,3,1,0,False
15,20445,2179.0,0.0,0.03859,0.00000,0.02270,0.00000,0.0,0.00114,0.00852,...,0.0,0.0,0.0,0.0,Room AC,2,4,1,0,False
19,20445,2179.0,0.0,0.00000,0.00000,0.02214,0.00000,0.0,0.00105,0.00608,...,70.0,0.0,0.0,0.0,Room AC,2,5,1,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007983,9685,1228.0,0.0,0.00000,0.00565,0.00000,0.00000,0.0,0.00146,0.03316,...,0.0,0.0,0.0,0.0,Room AC,1,20,12,0,False
7007987,9685,1228.0,0.0,0.00000,0.00000,0.00000,0.00000,0.0,0.00186,0.04399,...,0.0,0.0,0.0,0.0,Room AC,1,21,12,0,False
7007991,9685,1228.0,0.0,0.00000,0.00000,0.00000,0.00000,0.0,0.00212,0.05232,...,0.0,0.0,0.0,0.0,Room AC,1,22,12,0,False
7007995,9685,1228.0,0.0,0.00000,0.00000,0.00000,0.00000,0.0,0.00226,0.05078,...,0.0,0.0,0.0,0.0,Room AC,1,23,12,0,False


In [6]:
data_final.to_parquet('final_data.parquet')

In [7]:
for col in data_final.columns:
    print(col)

building_id
sqft
elec_ceiling_fan_kwh
elec_clothes_washer_kwh
elec_cooling_kwh
elec_freezer_kwh
elec_heating_kwh
elec_hot_water_kwh
elec_lighting_exterior_kwh
elec_lighting_interior_kwh
elec_plug_loads_kwh
elec_range_oven_kwh
elec_refrigerator_kwh
elec_television_kwh
weather_drybulb_temp_c
weather_relative_humidity_pct
weather_wind_speed_m_s
weather_wind_direction_deg
weather_global_horizontal_radiation_w_m2
weather_direct_normal_radiation_w_m2
weather_diffuse_horizontal_radiation_w_m2
hvac_cooling_type
bedrooms
hour
month
day_of_week
is_weekend
